# Huấn luyện mô hình HAR và Phát hiện Té ngã (CNN-LSTM)

Notebook này được thiết kế để chạy trực tiếp trên môi trường **Google Colab**.

### Hướng dẫn sử dụng trên Colab:
1. **Tải lên dữ liệu:** Bạn nén toàn bộ thư mục `SisFall_dataset_Windowed` trên máy tính của bạn thành file `SisFall_dataset_Windowed.zip` và upload lên thư mục gốc của **Google Drive**.
2. **Chạy Notebook:** Chạy lần lượt các block dưới đây. Nó sẽ tự động kết nối Drive, giải nén dữ liệu vào ổ cứng siêu tốc của Colab (`/content/`), nạp dữ liệu và tiến hành huấn luyện bằng GPU.
3. **Lưu Model:** Model tốt nhất sẽ được tự động lưu về Google Drive của bạn ở dạng file `best_model_har.keras` để sau này bạn dùng cho TinyML.

In [ ]:
# 1. Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Giải nén dữ liệu từ Google Drive vào ổ cứng cục bộ của Colab
# Lưu ý: Đường dẫn ZIP_PATH này giả định bạn ném file ZIP thẳng vào thư mục gốc của My Drive
import os

ZIP_PATH = '/content/drive/MyDrive/SisFall_dataset_Windowed.7z'
EXTRACT_PATH = '/content/SisFall_dataset_Windowed'

if not os.path.exists(EXTRACT_PATH):
    print("Đang giải nén dữ liệu...")
    !7z x "$ZIP_PATH" -o/content/ > /dev/null
    print("Giải nén hoàn tất!")
else:
    print("Thư mục đã được giải nén sẵn!")

In [ ]:
# 3. Khai báo thư viện
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow Version:", tf.__version__)

# Kiểm tra GPU
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
  print("[*] Đang sử dụng GPU:", gpu_devices)
else:
  print("[*] Cảnh báo: Đang chạy bằng CPU!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Cấu hình biến môi trường và LSO
DATA_DIR = Path(EXTRACT_PATH)

LABEL_MAP = {
    'D01': 0, 'D02': 0, 'D05': 0, 'D06': 0, 'D19': 0,
    'D03': 1, 'D04': 1,
    'D07': 2, 'D08': 2, 'D09': 2, 'D10': 2, 'D11': 2, 'D12': 2, 'D13': 2, 'D14': 2, 'D15': 2, 'D16': 2, 'D17': 2, 'D18': 2,
    'F01': 3, 'F02': 3, 'F03': 3, 'F04': 3, 'F05': 3, 'F06': 3, 'F07': 3, 'F08': 3, 'F09': 3,
    'F10': 3, 'F11': 3, 'F12': 3, 'F13': 3, 'F14': 3, 'F15': 3
}
CLASS_NAMES = ['Walk', 'Run', 'Static/ADL', 'Fall']

# Chia tập Leave-Subjects-Out
TRAIN_SUBJECTS = {f"SA{i:02d}" for i in range(1, 19)} | {f"SE{i:02d}" for i in range(1, 9)}
VAL_SUBJECTS = {f"SA{i:02d}" for i in range(19, 22)} | {f"SE{i:02d}" for i in range(9, 12)}
TEST_SUBJECTS = {f"SA{i:02d}" for i in range(22, 24)} | {f"SE{i:02d}" for i in range(12, 16)}

In [ ]:
# 5. Hàm tải và phân bổ dữ liệu CSV
def parse_filename_info(filename):
    parts = filename.split('_')
    return parts[0], parts[1]

def load_single_csv(file_path):
    try:
        label_code, subject_id = parse_filename_info(file_path.stem)
        if label_code not in LABEL_MAP: return None
        df = pd.read_csv(file_path)
        if len(df) != 200: return None
        return df.to_numpy(), LABEL_MAP[label_code], subject_id
    except:
        return None

def prepare_dataset():
    all_files = list(DATA_DIR.rglob('*.csv'))
    total_files = len(all_files)
    print(f"[*] Tìm thấy {total_files} file. Đang tải dữ liệu song song (Sử dụng 16 threads)...")

    X_train_list, y_train_list = [], []
    X_val_list, y_val_list = [], []
    X_test_list, y_test_list = [], []

    processed_count = 0
    with ThreadPoolExecutor(max_workers=16) as executor:
        for res in executor.map(load_single_csv, all_files):
            processed_count += 1
            if processed_count % 10000 == 0:
                print(f"  > Đã quét {processed_count}/{total_files} file...")
            if res is None: continue

            data, label, subject_id = res
            if subject_id in TRAIN_SUBJECTS:
                X_train_list.append(data); y_train_list.append(label)
            elif subject_id in VAL_SUBJECTS:
                X_val_list.append(data); y_val_list.append(label)
            elif subject_id in TEST_SUBJECTS:
                X_test_list.append(data); y_test_list.append(label)

    X_train = np.array(X_train_list, dtype=np.float32)
    y_train = np.array(y_train_list, dtype=np.int32)
    X_val = np.array(X_val_list, dtype=np.float32)
    y_val = np.array(y_val_list, dtype=np.int32)
    X_test = np.array(X_test_list, dtype=np.float32)
    y_test = np.array(y_test_list, dtype=np.int32)

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = prepare_dataset()
print(f"\n[*] Train shape: {X_train.shape}, Nhãn: {np.bincount(y_train)}")
print(f"[*] Val shape:   {X_val.shape}, Nhãn: {np.bincount(y_val)}")
print(f"[*] Test shape:  {X_test.shape}, Nhãn: {np.bincount(y_test)}")

In [ ]:
# 6. Xây dựng mô hình CNN-LSTM
def build_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        LSTM(128, return_sequences=True),
        LSTM(64),
        Dropout(0.4),

        Dense(32, activation='relu'),
        Dense(4, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_model((X_train.shape[1], X_train.shape[2]))
model.summary()

# Tính toán trọng số lớp cân bằng (Xử lý Imbalance Data)
cw_vals = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(zip(np.unique(y_train), cw_vals))
print("\n[*] Class Weights:", class_weights)

In [ ]:
# 7. Huấn luyện mô hình siêu tốc
# Lưu trữ checkpoint tốt nhất trực tiếp lên Google Drive
model_path = '/content/drive/MyDrive/best_model_har.keras'

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ModelCheckpoint(model_path, monitor='val_loss', save_best_only=True, verbose=1)
]

print("\n[*] Bắt đầu huấn luyện...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=256, class_weight=class_weights, callbacks=callbacks, verbose=1
)

In [ ]:
# 8. Đánh giá kết quả trên tập Test chưa từng học (LSO)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.show()

# Tải mô hình tốt nhất từ Drive
best_model = tf.keras.models.load_model(model_path)
y_pred = np.argmax(best_model.predict(X_test, batch_size=256), axis=1)

print("\n================ REPORT =================")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix trên Test Set')
plt.colorbar()
tick_marks = np.arange(len(CLASS_NAMES))
plt.xticks(tick_marks, CLASS_NAMES, rotation=45)
plt.yticks(tick_marks, CLASS_NAMES)
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'), horizontalalignment="center", color="white" if cm[i, j] > thresh else "black")
plt.ylabel('Thực Tế')
plt.xlabel('Dự Đoán')
plt.show()

# Đánh giá chuyên sâu Recall của té ngã
true_falls = np.sum(y_test == 3)
detected_falls = cm[3, 3]
recall_fall = (detected_falls / true_falls) * 100 if true_falls > 0 else 0
print(f"\n[!!!] TỶ LỆ RECALL LỚP TÉ NGÃ (FALL): {recall_fall:.2f}%")
if recall_fall > 95:
    print("    => TUYỆT VỜI! Đạt chỉ tiêu trên 95% phát hiện ngã!")
else:
    print("    => Cần thu thập thêm dữ liệu hoặc Data Augmentation.")